In [2]:
# Librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import scipy.optimize as sco

from itertools import combinations
from pathlib import Path

# Paleta de colores BBVA
paleta_bbva = {
    "azul_logotipo": "#004481",
    "azul_oscuro_digital": "#072146",
    "azul_celeste_aqua": "#5BBEFF",
    "azul_medio": "#1973B8",
    "azul_claro": "#D4EDFC"
}

colores_bbva_list = list(paleta_bbva.values())

In [3]:
# BASE_DIR apunta a la carpeta CODIGO
BASE_DIR = Path.cwd()

# Cargar df_diario.parquet desde la raíz del proyecto
ruta_archivo = BASE_DIR.parent / "df_diario.parquet"

df = pd.read_parquet(ruta_archivo)

df.head()

Ticker           AAPL                                                AMGN  \
Atributo        close       high        low       open volume       close   
date                                                                        
2018-06-06  45.890783  45.890783  45.405855  45.803260    0.0  115.655362   
2018-06-07  45.772508  45.928631  45.508754  45.927449    0.0  115.497849   
2018-06-08  45.301773  45.416500  44.898454  45.176401    0.0  116.127938   
2018-06-11  45.218980  45.410586  44.998988  45.262742    0.0  116.323276   
2018-06-12  45.493378  45.559613  45.222528  45.272204    0.0  116.613127   

Ticker                                                     ...         UNH  \
Atributo          high         low        open     volume  ...       close   
date                                                       ...               
2018-06-06  116.033418  113.966681  114.684998  2213100.0  ...  193.867722   
2018-06-07  115.970426  115.000062  115.604965  2729600.0  ...  194.125062   
2018-06-08  116.978571  115.686868  115.869594  2865000.0  ...  195.450467   
2018-06-11  116.455602  115.264699  116.027127  2409400.0  ...  197.867518   
2018-06-12  116.644625  115.567154  116.109043  2045300.0  ...  197.711575   

Ticker                                                              V  \
Atributo          high         low        open     volume       close   
date                                                                    
2018-06-06  193.922305  191.123252  192.051075  2650300.0  122.520808   
2018-06-07  194.920341  193.555898  194.733213  2117200.0  120.327169   
2018-06-08  196.089811  193.415510  194.499267  3034600.0  121.136333   
2018-06-11  198.810938  195.232192  195.575254  2775200.0  128.074994   
2018-06-12  197.992261  195.052859  197.415292  2567200.0  128.341777   

Ticker                                                     
Atributo          high         low        open     volume  
date                                                       
2018-06-06  122.547778  120.120380  120.471004  8065800.0  
2018-06-07  122.790537  119.616936  122.592748  8309200.0  
2018-06-08  121.298153  119.778778  119.967582  5261000.0  
2018-06-11  128.784828  127.808211  128.437057        0.0  
2018-06-12  128.713368  127.236533  127.660527        0.0  

[5 rows x 130 columns]

In [4]:
# Extraer y limpiar nombres de columnas
df_close = df[[c for c in df.columns if c[1] == "close"]].copy()
df_close.columns = [c[0] for c in df_close.columns]

# Graficar evolución de los precios de cierre
fig = px.line(
    df_close,
    title="Evolución del Precio de Cierre",
    labels={"value": "Precio", "index": "Fecha"},
    color_discrete_sequence=colores_bbva_list
)

fig.show()

In [5]:
# 1. Preparar retornos
activos = ['AXP', 'CRM', 'MSFT', 'KO', 'JNJ', 'GS', 'PG', 'DIS', 'UNH', 'MRK', 'V', 'CVX', 'BA', 'INTC', 'MMM', 'IBM', 'CSCO', 'AAPL', 'MCD', 'CAT', 'HD', 'AMGN', 'JPM', 'NKE', 'TRV', 'DOW']
retornos = df.xs("close", axis=1, level=1).filter(items=activos).pct_change().dropna()

r_anual = retornos.mean() * 252
cov_anual = retornos.cov() * 252
matriz_corr = retornos.corr()

# 2. Calcular métricas para cada trío
resultados = []
for trio in combinations(retornos.columns, 3):
    t = list(trio)
    rent = r_anual[t].mean()
    vol = np.sqrt(cov_anual.loc[t, t].values.sum() / 9)
    corr_media = (matriz_corr.loc[t, t].values.sum() - 3) / 6
    
    resultados.append({
        "Trio": " - ".join(t), 
        "Corr": round(corr_media, 4), 
        "Sharpe": round(rent / vol, 4), 
        "Rent": rent, 
        "Vol": vol
    })

df_t = pd.DataFrame(resultados)

# 3. Identificar el mejor trío
top = df_t.loc[(df_t["Sharpe"] - df_t["Corr"]).idxmax()]
opt_label = f"ÓPTIMO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_label, "Otros")

# 4. Crear gráfico
fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Rent", "Vol"],
    color_discrete_map={"Otros": paleta_bbva["azul_celeste_aqua"], opt_label: paleta_bbva["azul_oscuro_digital"]},
    title="Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>Sharpe: {top['Sharpe']:.2f} | Corr: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-80, ay=-30, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_label))
fig.update_layout(height=700, width=850).show()

In [6]:
cartera_tickers = ['MSFT', 'PG', 'CAT']
matriz_corr = retornos[cartera_tickers].corr().round(4)
print(matriz_corr)

        MSFT      PG     CAT
MSFT  1.0000  0.3619  0.2386
PG    0.3619  1.0000  0.0853
CAT   0.2386  0.0853  1.0000


In [7]:
# 1. Preparar datos de retornos
df_close = df.xs("close", axis=1, level=1).filter(items=activos)
df_close.index = pd.to_datetime(df[("date", "date")] if ("date", "date") in df.columns else df.get("date", df.index))
ret = df_close.pct_change().dropna()

r_d, cov_d, corr_m = ret.mean(), ret.cov(), ret.corr()
w = np.array([1/3, 1/3, 1/3])
resultados = []

# 2. Filtrar tríos con correlación asimétrica
for trio in combinations(ret.columns, 3):
    t = list(trio)
    c_sub = corr_m.loc[t, t].values
    r12, r13, r23 = c_sub[0, 1], c_sub[0, 2], c_sub[1, 2]
    _, r_min2, r_max = sorted([r12, r13, r23])
    
    if r_max >= 0.50 and r_min2 <= 0.25:
        rent = np.dot(r_d[t], w) * 252
        vol = np.sqrt(w.T @ cov_d.loc[t, t] @ w) * np.sqrt(252)
        
        if r_max == r12: bloque = f"{t[0]}-{t[1]} + {t[2]}"
        elif r_max == r13: bloque = f"{t[0]}-{t[2]} + {t[1]}"
        else: bloque = f"{t[1]}-{t[2]} + {t[0]}"
            
        resultados.append({
            "Trio": " - ".join(t), "Estructura": bloque,
            "Corr_Max_Par": round(r_max, 4), "Corr": round((r12 + r13 + r23) / 3, 4),
            "Sharpe": round(rent / vol, 4), "Rent": rent, "Vol": vol
        })

df_t = pd.DataFrame(resultados)

# 3. Identificar el óptimo y graficar
top = df_t.loc[df_t["Sharpe"].idxmax()]
opt_cat = f"ÓPTIMO ASIMÉTRICO: {top['Trio']}"
df_t["Cat"] = np.where(df_t.index == top.name, opt_cat, "Otros Asimétricos")

fig = px.scatter(
    df_t, x="Corr", y="Sharpe", color="Cat", hover_data=["Trio", "Estructura", "Corr_Max_Par", "Rent", "Vol"],
    color_discrete_map={"Otros Asimétricos": paleta_bbva["azul_celeste_aqua"], opt_cat: paleta_bbva["azul_oscuro_digital"]},
    title="Estrategia Asimétrica: Sharpe vs Correlación",
    labels={"Corr": "Correlación Media", "Sharpe": "Ratio Sharpe"}
)

fig.add_annotation(
    x=top["Corr"], y=top["Sharpe"], 
    text=f"<b>{top['Trio']}</b><br>{top['Estructura']}<br>Sharpe: {top['Sharpe']:.2f} | Corr Media: {top['Corr']:.2f}",
    showarrow=True, arrowhead=2, arrowcolor=paleta_bbva["azul_logotipo"], ax=-120, ay=-40, font=dict(color=paleta_bbva["azul_logotipo"])
)

fig.update_traces(marker=dict(size=5, opacity=0.4), selector=dict(name="Otros Asimétricos"))
fig.update_traces(marker=dict(size=14, symbol="star"), selector=dict(name=opt_cat))
fig.update_layout(height=700, width=850).show()

In [8]:
cartera_tickers = ['GS', 'PG', 'V']
matriz_corr = ret[cartera_tickers].corr().round(4)
print(matriz_corr)

        GS      PG       V
GS  1.0000  0.1458  0.5441
PG  0.1458  1.0000  0.2506
V   0.5441  0.2506  1.0000


In [9]:
# Frontera eficiente: tres activos y pesos de 5 %

distribuciones = np.array([
    [i, j, 20-i-j]
    for i in range(1, 19)
    for j in range(1, 20-i)
]) / 20

resultados = []

for trio in combinations(retornos.columns, 3):
    t = list(trio)
    pesos = distribuciones
    rent = pesos @ r_anual[t].values
    cov = cov_anual.loc[t, t].values
    vol = np.sqrt(np.maximum(np.einsum("ij,jk,ik->i", pesos, cov, pesos), 0))

    resultados.extend([
        {"Trio": " - ".join(t), "Rent": r, "Vol": v, "Sharpe": r/v, "Pesos": w}
        for r, v, w in zip(rent, vol, pesos)
    ])

df_carteras = pd.DataFrame(resultados)
mejor_libre = df_carteras.loc[df_carteras["Sharpe"].idxmax()]

# Frontera eficiente
orden = df_carteras.sort_values(["Vol", "Rent"], ascending=[True, False])
df_frontera_libre = orden[
    orden["Rent"] > orden["Rent"].cummax().shift(fill_value=-np.inf) + 1e-8
]

# Etiqueta de la cartera óptima
activos_optimos = mejor_libre["Trio"].split(" - ")
composicion = "<br>".join(
    f"{a}: {p:.0%}" for a, p in zip(activos_optimos, mejor_libre["Pesos"])
)

etiqueta = (
    f"<b>CARTERA ÓPTIMA</b><br>{mejor_libre['Trio']}<br>"
    f"Rentabilidad: {mejor_libre['Rent']:.2%}<br>"
    f"Volatilidad: {mejor_libre['Vol']:.2%}<br>"
    f"Sharpe: {mejor_libre['Sharpe']:.3f}<br>{composicion}"
)

# Gráfico
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_carteras["Vol"], y=df_carteras["Rent"],
    mode="markers", name="Carteras",
    marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=4, opacity=0.35)
))

fig.add_trace(go.Scatter(
    x=df_frontera_libre["Vol"], y=df_frontera_libre["Rent"],
    mode="lines", name="Frontera eficiente",
    line=dict(color=paleta_bbva["azul_logotipo"], width=2.5)
))

fig.add_trace(go.Scatter(
    x=[mejor_libre["Vol"]], y=[mejor_libre["Rent"]],
    mode="markers", name="Cartera óptima",
    marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=17, symbol="star")
))

fig.add_annotation(
    x=mejor_libre["Vol"], y=mejor_libre["Rent"], text=etiqueta,
    showarrow=True, ax=100, ay=-90,
    bgcolor="white", bordercolor=paleta_bbva["azul_logotipo"],
    borderpad=7, font=dict(size=11)
)

fig.update_layout(
    title="Frontera Eficiente - Tres Activos sin Restricción de Pesos",
    xaxis_title="Volatilidad Anualizada",
    yaxis_title="Rentabilidad Anualizada",
    xaxis_tickformat=".0%", yaxis_tickformat=".0%",
    width=1100, height=750
)

fig.show()

print("Cartera óptima:", mejor_libre["Trio"])
print(f"Rentabilidad: {mejor_libre['Rent']:.2%} | Volatilidad: {mejor_libre['Vol']:.2%}")
print(f"Sharpe: {mejor_libre['Sharpe']:.3f}")
print(composicion.replace("<br>", "\n"))
print(f"Total de carteras analizadas: {len(df_carteras):,}")

Cartera óptima: MSFT - GS - AAPL
Rentabilidad: 28.24% | Volatilidad: 25.26%
Sharpe: 1.118
MSFT: 60%
GS: 35%
AAPL: 5%
Total de carteras analizadas: 444,600


In [10]:
# Frontera eficiente: tres activos con inversión a partes iguales

resultados = []

for trio in combinations(retornos.columns, 3):
    t = list(trio)
    pesos = np.array([1/3, 1/3, 1/3])

    rent = np.dot(r_anual[t], pesos)
    cov = cov_anual.loc[t, t].values
    vol = np.sqrt(pesos @ cov @ pesos)

    resultados.append({
        "Trio": " - ".join(t),
        "Rent": rent,
        "Vol": vol,
        "Sharpe": rent / vol,
        "Pesos": pesos
    })

df_carteras_iguales = pd.DataFrame(resultados)

# Cartera óptima
mejor_igual = df_carteras_iguales.loc[
    df_carteras_iguales["Sharpe"].idxmax()
]

# Frontera eficiente
orden = df_carteras_iguales.sort_values(
    ["Vol", "Rent"], ascending=[True, False]
)

df_frontera_igual = orden[
    orden["Rent"] > orden["Rent"].cummax().shift(fill_value=-np.inf) + 1e-8
]

# Etiqueta de la cartera óptima
etiqueta = (
    f"<b>CARTERA ÓPTIMA</b><br>{mejor_igual['Trio']}<br>"
    f"Rentabilidad: {mejor_igual['Rent']:.2%}<br>"
    f"Volatilidad: {mejor_igual['Vol']:.2%}<br>"
    f"Sharpe: {mejor_igual['Sharpe']:.3f}<br>"
    f"Pesos: 33,33 % cada activo"
)

# Gráfico
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_carteras_iguales["Vol"], y=df_carteras_iguales["Rent"],
    mode="markers", name="Carteras",
    marker=dict(color=paleta_bbva["azul_celeste_aqua"], size=6, opacity=0.5)
))

fig.add_trace(go.Scatter(
    x=df_frontera_igual["Vol"], y=df_frontera_igual["Rent"],
    mode="lines+markers", name="Frontera eficiente",
    line=dict(color=paleta_bbva["azul_logotipo"], width=2.5)
))

fig.add_trace(go.Scatter(
    x=[mejor_igual["Vol"]], y=[mejor_igual["Rent"]],
    mode="markers", name="Cartera óptima",
    marker=dict(color=paleta_bbva["azul_oscuro_digital"], size=17, symbol="star")
))

fig.add_annotation(
    x=mejor_igual["Vol"], y=mejor_igual["Rent"], text=etiqueta,
    showarrow=True, ax=100, ay=-90,
    bgcolor="white", bordercolor=paleta_bbva["azul_logotipo"],
    borderpad=7, font=dict(size=11)
)

fig.update_layout(
    title="Frontera Eficiente - Tres Activos con Pesos Iguales",
    xaxis_title="Volatilidad Anualizada",
    yaxis_title="Rentabilidad Anualizada",
    xaxis_tickformat=".0%", yaxis_tickformat=".0%",
    width=1100, height=750
)

fig.show()

# Resultados
print("CARTERA ÓPTIMA")
print("-----------------------------")
print("Activos:", mejor_igual["Trio"])
print(f"Rentabilidad: {mejor_igual['Rent']:.2%}")
print(f"Volatilidad: {mejor_igual['Vol']:.2%}")
print(f"Sharpe: {mejor_igual['Sharpe']:.3f}")
print(f"\nTotal de carteras analizadas: {len(df_carteras_iguales):,}")

CARTERA ÓPTIMA
-----------------------------
Activos: MSFT - GS - PG
Rentabilidad: 22.08%
Volatilidad: 20.51%
Sharpe: 1.077

Total de carteras analizadas: 2,600


In [11]:
# Selección de carteras con criterios de correlación, riesgo bajista y COVID

rf_diario = 0.03 / 252

# Correlaciones móviles de 80 días
rolling_corr = retornos.rolling(80).corr().dropna()
corr_media = rolling_corr.groupby(level=1).mean()

fechas = rolling_corr.index.get_level_values(0).unique()
fechas_covid = fechas[
    (fechas >= "2020-02-19") & (fechas <= "2020-11-09")
]

candidatas = []

for _, c in df_frontera_igual.iterrows():
    t = c["Trio"].split(" - ")

    # Ordenar los pares según su correlación media
    pares = sorted([
        (corr_media.loc[a, b], a, b)
        for a, b in combinations(t, 2)
    ])

    # Correlación media de los dos pares menos correlacionados durante COVID
    medias_covid = [
        rolling_corr.xs(a, level=1)[b].loc["2020-02-19":"2020-11-09"].mean()
        for _, a, b in pares[:2]
    ]

    # Rendimiento diario de la cartera a partes iguales
    rent_diaria = retornos[t].mean(axis=1)

    # Riesgo bajista y ratio de Sortino
    downside = np.minimum(0, rent_diaria - rf_diario)
    vol_negativa = np.sqrt(np.mean(downside**2)) * np.sqrt(252)

    sortino = (c["Rent"] - 0.03) / vol_negativa

    candidatas.append({
        "Trio": c["Trio"],
        "Rent": c["Rent"],
        "Vol_Negativa": vol_negativa,
        "Sortino": sortino,
        "Score_Crisis": np.mean(medias_covid),
        "Corr_Alta": pares[2][0],
        "Corr_Baja1": pares[0][0],
        "Corr_Baja2": pares[1][0]
    })

df_cand = pd.DataFrame(candidatas)

# Aplicar los criterios de correlación
df_cand = df_cand[
    (df_cand["Corr_Alta"] >= 0.45) &
    (df_cand["Corr_Baja1"] < 0.35) &
    (df_cand["Corr_Baja2"] < 0.35) &
    (df_cand["Score_Crisis"].notna())
].copy()

# Agrupar por rentabilidad y seleccionar el mejor Sortino
df_cand = df_cand.sort_values("Vol_Negativa")

df_cand["Grupo"] = pd.qcut(
    df_cand["Rent"],
    q=min(8, df_cand["Rent"].nunique()),
    duplicates="drop"
)

frontera_final = (
    df_cand.loc[
        df_cand.groupby("Grupo", observed=True)["Sortino"].idxmax()
    ]
    .sort_values("Score_Crisis")
    .reset_index(drop=True)
)

# Mostrar resultados en texto plano
print(frontera_final.to_string(index=False))

             Trio     Rent  Vol_Negativa  Sortino  Score_Crisis  Corr_Alta  Corr_Baja1  Corr_Baja2          Grupo
MSFT - AAPL - CAT 0.379343      0.195366 1.788151      0.082690   0.590843    0.228305    0.231378 (0.242, 0.379]
   PG - MCD - CAT 0.167489      0.138084 0.995694      0.110178   0.482940    0.085340    0.199778 (0.161, 0.167]
    JNJ - GS - PG 0.160705      0.134821 0.969471      0.164752   0.486041    0.145093    0.215245 (0.141, 0.161]
 MSFT - GS - AAPL 0.389017      0.198947 1.804586      0.215169   0.590843    0.304556    0.322158 (0.379, 0.389]
 MSFT - CAT - JPM 0.242240      0.165017 1.286170      0.317723   0.520972    0.231378    0.284598 (0.202, 0.242]
    MSFT - PG - V 0.201804      0.136830 1.255605      0.321461   0.450604    0.268146    0.285255 (0.185, 0.202]
   MSFT - JNJ - V 0.185461      0.133753 1.162297      0.343331   0.450604    0.179083    0.257404 (0.167, 0.185]
MSFT - IBM - AAPL 0.398078      0.219018 1.680587      0.433430   0.590843    0.259592  

Se selecciona la cartera compuesta por MSFT, GS y CAT